In [1]:
import sys
import platform
import datetime as dt
import pandas as pd
import numpy as np

# ============================================================
# Zweck
# ============================================================
# Wir vergleichen q = 0.90 / 0.95 / 0.99.
# Für jedes q:
#   1) berechne ID = |2*(read_pct/100) - 1|
#   2) setze eps = Quantil_q(ID) NUR aus Ransomware
#   3) markiere "ransomware_like" für alle Daten: ID <= eps
#   4) zähle:
#      - TP/FN auf Ransomware
#      - FP auf Benign+Encryption
# ============================================================


# ---------------------------
# Repro-Infos (für BA)
# ---------------------------
DATA_SOURCE_URL = "https://dr.lib.iastate.edu/server/api/core/bitstreams/56f4c518-61ec-43fb-8bee-2287e851e597/content"
TABLES_USED = {"benign": "Table 7.2", "encryption": "Table 7.4", "ransomware": "Table 7.5"}
QS = [0.90, 0.95, 0.99]
INTERP = "linear"  # Quantil-Interpolation explizit, damit reproduzierbar

print("=== REPRO INFO ===")
print("source:", DATA_SOURCE_URL)
print("tables:", TABLES_USED)
print("date:", dt.date.today().isoformat())
print("python:", sys.version.split()[0], "| pandas:", pd.__version__, "| numpy:", np.__version__)
print("os:", platform.system(), platform.release(), platform.machine())
print("settings:", {"qs": QS, "quantile_interpolation": INTERP})
print()


# ---------------------------
# Daten (Workload -> read_pct)
# ---------------------------
RANSOMWARE = {
    "ransomware_base64": 38.91, "ransomware_inplace": 54.26, "ransomware_outplace": 57.44,
    "ransomware_padding": 31.52, "Linux.Cryptor": 59.90, "Defray": 59.98, "hellokitty": 51.25,
    "Lockbit": 59.99, "Polaris": 53.07, "ransomexx": 65.56, "KeypassRansomware": 59.49,
    "LockCrypt2.0": 59.02,
}
BENIGN = {
    "pgbench": 17.36, "filebench web_server": 93.43, "filebench fileserver": 30.01,
    "filebench randrw": 43.21, "filebench webproxy": 86.75, "fio rand-rw": 50.77,
    "fio seq-rw": 66.48, "fio seq-read-medium-real": 66.48, "SNIA RocksDB SSD": 50.68,
}
ENCRYPTION = {
    "aes-128-cbc": 54.65, "aes-256-cbc": 17.49, "aes-256-gcm": 24.97, "chacha20-poly1305": 22.77,
}

# Dict -> DataFrame (damit klar ist, welcher Wert zu welchem Workload gehört)
def to_df(d, cls):
    return pd.Series(d, name="read_pct").rename_axis("workload").reset_index().assign(cls=cls)

df = pd.concat([to_df(RANSOMWARE, "ransomware"),
                to_df(BENIGN, "benign"),
                to_df(ENCRYPTION, "encryption")], ignore_index=True)

# Feature: ID aus read_pct
p = df["read_pct"] / 100.0

#  1 = read + write
# write = 1 - read =>  |read - write| = | 2*read -1|

# |read - write| / (read + write)
# |2*read -1| / 1 =
#
df["ID"] = (2*p - 1).abs()


# ---------------------------
# Auswertung für mehrere q
# ---------------------------
rows = []
for q in QS:
    # eps aus Ransomware (Baseline)
    eps = float(df.loc[df["cls"] == "ransomware", "ID"].quantile(q, interpolation=INTERP))

    # Regel auf ALLE Daten anwenden
    df_q = df.copy()
    df_q["ransomware_like"] = df_q["ID"] <= eps

    # Ransomware: TP/FN
    rw = df_q[df_q["cls"] == "ransomware"]
    tp = int(rw["ransomware_like"].sum())
    fn = int((~rw["ransomware_like"]).sum())

    # Non-Ransomware: FP
    non_r = df_q[df_q["cls"] != "ransomware"]
    fp = int(non_r["ransomware_like"].sum())
    total = int(len(non_r))
    fp_rate = fp / total

    rows.append({"q": q, "eps": eps, "TP": f"{tp}/{len(rw)}", "FN": f"{fn}/{len(rw)}",
                 "FP": f"{fp}/{total}", "FP_rate": fp_rate})

summary = pd.DataFrame(rows)

print("=== SUMMARY (q Vergleich) ===")
print(summary.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print()

# Optional (für Argumentation): zeige False-Positive Workloads für ein q
# -> hier Beispiel: q=0.90
q_show = 0.90
eps_show = float(df.loc[df["cls"] == "ransomware", "ID"].quantile(q_show, interpolation=INTERP))
df_show = df.copy()
df_show["ransomware_like"] = df_show["ID"] <= eps_show

fp_list = df_show[(df_show["cls"] != "ransomware") & (df_show["ransomware_like"])][["cls", "workload", "read_pct", "ID"]]
print(f"=== FALSE POSITIVES für q={int(q_show*100)} ===")
print(fp_list.sort_values(["cls", "workload"]).to_string(index=False, float_format=lambda x: f"{x:.6f}"))


=== REPRO INFO ===
source: https://dr.lib.iastate.edu/server/api/core/bitstreams/56f4c518-61ec-43fb-8bee-2287e851e597/content
tables: {'benign': 'Table 7.2', 'encryption': 'Table 7.4', 'ransomware': 'Table 7.5'}
date: 2026-02-25
python: 3.12.3 | pandas: 3.0.1 | numpy: 2.4.2
os: Linux 6.17.0-14-generic x86_64
settings: {'qs': [0.9, 0.95, 0.99], 'quantile_interpolation': 'linear'}

=== SUMMARY (q Vergleich) ===
       q      eps    TP   FN   FP  FP_rate
0.900000 0.302260 10/12 2/12 4/13 0.307692
0.950000 0.337480 11/12 1/12 6/13 0.461538
0.990000 0.363176 11/12 1/12 6/13 0.461538

=== FALSE POSITIVES für q=90 ===
       cls         workload  read_pct       ID
    benign SNIA RocksDB SSD 50.680000 0.013600
    benign filebench randrw 43.210000 0.135800
    benign      fio rand-rw 50.770000 0.015400
encryption      aes-128-cbc 54.650000 0.093000


### Definitionen
Sei $r = \text{read}$ und $w = \text{write}$, wobei $r + w = 1$.

$$
\begin{align}
w &= 1 - r \\
\frac{|r - w|}{r + w} &= \frac{|r - (1-r)|}{1} \\
&= |2r - 1|
\end{align}
$$